In [4]:
%matplotlib inline
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [5]:
def load_data_fashion_mnist(batch_size, resize=None):
    """替代 d2l.load_data_fashion_mnist"""
    trans = []

    if resize:
        trans.append(transforms.Resize(resize))

    trans.append(transforms.ToTensor())
    transform = transforms.Compose(trans)

    train_dataset = datasets.FashionMNIST(
        root="./data",
        train=True,
        transform=transform,
        download=True
    )

    test_dataset = datasets.FashionMNIST(
        root="./data",
        train=False,
        transform=transform,
        download=True
    )

    train_iter = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    test_iter = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return train_iter, test_iter

## 唯一新增加的东西

In [6]:
net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

In [7]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_weights)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=10, bias=True)
)

In [8]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)

    cmp = y_hat.type(y.dtype) == y
    return float(cmp.sum())

In [9]:
def evaluate_accuracy(net, data_iter):
    """计算模型在数据集上的准确率"""
    net.eval()

    correct = 0.0
    total = 0

    with torch.no_grad():
        for X, y in data_iter:
            y_hat = net(X)
            correct += accuracy(y_hat, y)
            total += y.numel()

    return correct / total

In [10]:
def train_epoch_ch3(net, train_iter, loss, trainer):
    """训练一个 epoch"""
    net.train()

    total_loss = 0.0
    total_correct = 0.0
    total_num = 0

    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)

        trainer.zero_grad()
        l.mean().backward()
        trainer.step()

        total_loss += float(l.sum())
        total_correct += accuracy(y_hat, y)
        total_num += y.numel()

    return total_loss / total_num, total_correct / total_num

In [11]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    """替代 d2l.train_ch3"""
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch_ch3(
            net,
            train_iter,
            loss,
            trainer
        )

        test_acc = evaluate_accuracy(net, test_iter)

        print(
            f'epoch {epoch + 1}, '
            f'loss {train_loss:.4f}, '
            f'train acc {train_acc:.3f}, '
            f'test acc {test_acc:.3f}'
        )

In [12]:
batch_size, lr, num_epochs = 256, 0.1, 10

loss = nn.CrossEntropyLoss(reduction='none')
trainer = torch.optim.SGD(net.parameters(), lr=lr)

train_iter, test_iter = load_data_fashion_mnist(batch_size)

train_ch3(
    net,
    train_iter,
    test_iter,
    loss,
    num_epochs,
    trainer
)

C:\Users\hp\AppData\Local\Temp\ipykernel_3256\76618424.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  total_loss += float(l.sum())


epoch 1, loss 1.0420, train acc 0.641, test acc 0.735
epoch 2, loss 0.5937, train acc 0.792, test acc 0.753
epoch 3, loss 0.5178, train acc 0.819, test acc 0.767
epoch 4, loss 0.4832, train acc 0.831, test acc 0.820
epoch 5, loss 0.4516, train acc 0.841, test acc 0.836
epoch 6, loss 0.4366, train acc 0.845, test acc 0.843
epoch 7, loss 0.4190, train acc 0.852, test acc 0.844
epoch 8, loss 0.4042, train acc 0.857, test acc 0.848
epoch 9, loss 0.3929, train acc 0.861, test acc 0.822
epoch 10, loss 0.3830, train acc 0.865, test acc 0.828
